# 朴素贝叶斯

https://www.runoob.com/ml/ml-naive-bayes.html

想象一下，你正在网上书店浏览，系统根据你之前购买过《三体》和《流浪地球》，向你推荐了《球状闪电》。这个猜你喜欢的功能背后，很可能就用到了我们今天要讲的 朴素贝叶斯（Naive Bayes） 算法。

朴素贝叶斯是一种基于 贝叶斯定理 的简单而高效的 概率分类算法。

朴素贝叶斯的核心思想是：通过已知的某些特征（比如你买过的书），来计算某个事件（比如你会喜欢另一本书）发生的概率，并选择概率最高的类别作为预测结果。

它的朴素（Naive）之处在于一个关键假设：所有特征之间是相互独立的。也就是说，在判断你是否喜欢《球状闪电》时，算法认为购买过《三体》和购买过《流浪地球》这两个特征对你的决策影响是互不相关的。虽然在现实中，特征之间常有联系，但这个简化的假设让计算变得非常高效，且在许多实际场景中（尤其是文本分类）效果出奇地好。

## 核心原理：贝叶斯定理

要理解朴素贝叶斯，必须先了解它的基石——贝叶斯定理。它描述了在已知一些条件的情况下，如何更新某个事件发生的概率。

$$
P(A|B) = \frac{P(A) P(B|A)}{P(B)}
$$

一个例子来理解：场景：判断一封邮件是否是垃圾邮件（Spam）。

- A：邮件是垃圾邮件的事件
- B：邮件包含免费这个词的特征
- P(A)：任意一封邮件是垃圾邮件的 先验概率（比如，根据历史数据，100封邮件里有20封是垃圾邮件，那么 P(垃圾邮件) = 0.2）。
- P(B|A)：在已知邮件是垃圾邮件的情况下，其中出现"免费"这个词的 条件概率（比如，垃圾邮件中80%都包含"免费"，那么 P(免费|垃圾邮件) = 0.8）。
- P(B)：任意一封邮件中出现"免费"这个词的 总概率。
- P(A|B)： 我们最终想求的，在 已知邮件包含"免费"这个词 的条件下，这封邮件是垃圾邮件的 后验概率。

贝叶斯定理的精髓：它利用了我们已经知道的信息（垃圾邮件的普遍规律 P(A) 和垃圾邮件用词习惯 P(B|A)），结合新观察到的证据（这封邮件里有"免费"），来修正我们对这个具体事件的判断（这封邮件是垃圾邮件的可能性 P(A|B)）。

真正的贝叶斯分类器在计算 P(B|A) 时，需要考虑所有特征（B1， B2， B3...）的联合概率 P(B1， B2， B3... | A)，这非常复杂。

朴素贝叶斯做出了一个强大的简化假设：所有特征都相互条件独立。这意味着：

P(B1， B2， B3... | A) ≈ P(B1|A) * P(B2|A) * P(B3|A) * ...
这个假设将复杂的联合概率计算，简化成了多个简单概率的乘法，极大地降低了计算成本。

## 工作流程与分类器类型

## 动手实践：用 Python 实现垃圾邮件分类

In [3]:
# 示例训练数据：每行是一条邮件内容，后面是标签（'spam' 或 'ham'）
train_data = [
    ("免费获取 iPhone 大奖！点击链接", "spam"),
    ("老板，下午三点开会，请准时参加", "ham"),
    ("恭喜您中奖了！立即领取您的奖金", "spam"),
    ("项目报告已发到您的邮箱，请查收", "ham"),
    ("限时特价，全场五折，仅限今天", "spam"),
    ("周末聚餐定在晚上七点，老地方", "ham")
]

texts = [data[0] for data in train_data]  # 邮件文本列表
labels = [data[1] for data in train_data] # 对应标签列表

texts, labels

(['免费获取 iPhone 大奖！点击链接',
  '老板，下午三点开会，请准时参加',
  '恭喜您中奖了！立即领取您的奖金',
  '项目报告已发到您的邮箱，请查收',
  '限时特价，全场五折，仅限今天',
  '周末聚餐定在晚上七点，老地方'],
 ['spam', 'ham', 'spam', 'ham', 'spam', 'ham'])

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
import numpy as np

In [5]:
model = make_pipeline(CountVectorizer(), MultinomialNB())
model.fit(texts, labels)

,steps,"[('countvectorizer', ...), ('multinomialnb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [7]:
new_emails = [
    "免费领取优惠券，机会难得！",  # 预期为 spam
    "明天上午十点电话会议讨论预算"   # 预期为 ham
]

predictions = model.predict(new_emails)
prediction_proba = model.predict_proba(new_emails) # 获取预测概率

class_names = model.classes_
for email, pred, proba in zip(new_emails, predictions, prediction_proba):
    print(f'邮件内容: "{email}"')
    print(f"  预测类别: {pred}")
    for cls, prob in zip(class_names, proba):
        print(f"  属于'{cls}'的概率: {prob:.4f}")
    print("-" * 40)

邮件内容: "免费领取优惠券，机会难得！"
  预测类别: ham
  属于'ham'的概率: 0.5000
  属于'spam'的概率: 0.5000
----------------------------------------
邮件内容: "明天上午十点电话会议讨论预算"
  预测类别: ham
  属于'ham'的概率: 0.5000
  属于'spam'的概率: 0.5000
----------------------------------------


In [3]:
from sklearn.naive_bayes import MultinomialNB
import numpy as np

X = np.array([[1, 2], [2, 1], [3, 4], [4, 3]])
y = np.array([0, 0, 1, 1])
clf = MultinomialNB(alpha=1.0)
clf.fit(X, y)
print(clf.feature_log_prob_)
print(clf.class_log_prior_)

[[-0.69314718 -0.69314718]
 [-0.69314718 -0.69314718]]
[-0.69314718 -0.69314718]


## 手写 MultinomialNB

贝叶斯公式：
$$
P(Y \mid X) = \frac{P(X \mid Y) P(Y)}{P(X)}
$$

对我们这里的 $X$ 是由多个变量 $(x_0, x_1, \cdots )$ 组成，展开：
$$
P(Y\mid X) = \frac{  \prod_i P(x_i\mid Y) ^{x_i} P(Y)  }{P(X)}
$$

解释下，MultinomialNB 训练的输入应该是：

- $X$: (n_sample, n_features)：n_sample 个样本，每个样本 n_features 个整数，$i$ 个整数表示特征 $i$ 出现的次数（比如某个词出现的次数）
- $y$: (n_samples)：n_sample 个样本，每个样本的分类结果

MultinomialNB 先收集这样一批的输入，根据这些分布计算频率作为概率：

- $P(Y_i)$：属于第 $i$ 个分类的频率
- $P(x_i | Y_i)$：统计属于 $Y_i$ 的所有样本，出现特征 $x_i$ 的概率

保存这些值，predit 函数接受若干数据(n_samples, n_features)，对每个样本，根据其特征分布，可以计算出现这样样本特征的概率，然后对每个 $Y_i$，根据 贝叶斯公式 得到最大的概率的 $Y_i$。

所以对
$$
P(Y\mid X) = \frac{  \prod_i P(x_i\mid Y) ^{x_i} P(Y)  }{P(X)}
$$
公式，分子都是一样的，我们只要比较分母，并且 $\ln$ 函数是单调的，可以简化计算：
$$
\ln P(Y\mid X) =  \ln P(Y)  + \sum   {x_i} \ln P(x_i\mid Y)
$$

解释下这个 $\prod_i P(x_i\mid Y) ^{x_i}$，$P(x_i\mid Y)$ 表示 $i$ 特征出现的概率，而我们出现了 $x_i$ 次，所以使用次方关系。

In [30]:
import numpy as np
import math


class MyMultinomialNB:
    def __init__(self, alpha):
        self.alpha = alpha

    def fit(self, X, y):
        """
        X: (n_samples, n_features)
        y: (n_samples)
        """
        n_samples, n_features = X.shape
        n_class = int(np.max(y).item()) + 1

        # 统计每个分类出现的概率
        self.class_log_prior_ = np.zeros(n_class)
        for i in range(n_class):
            mask = y == i # (n_samples,)
            self.class_log_prior_[i] = math.log(np.sum(mask).item() / n_samples)

        # 统计每个分类
        self.feature_log_prob_ = np.zeros((n_class, n_features))
        for i in range(n_class):
            mask = y == i # (n_samples,)
            # 选择出属于 i 的 X
            X_i = X[mask] # (m, n_features)
            # 统计每个 features 出现的次数
            features_count = np.sum(X_i, axis=0) + self.alpha # (n_features)
            # 总的特征数
            all_features = np.sum(features_count)
            self.feature_log_prob_[i] = np.log(features_count / all_features)

    def predict(self, X):
        """
        X: (n_samples, n_features)
        """
        # (n_samples, n_features) @ (n_features, n_class) + (1, n_class)
        log_probs = X @ self.feature_log_prob_.T + self.class_log_prior_

        # 选概率最大的类别
        return np.argmax(log_probs, axis=1)

In [31]:
X = np.array([[1, 2], [2, 1], [3, 4], [4, 3]])
y = np.array([0, 0, 1, 1])
clf = MyMultinomialNB(alpha=1.0)
clf.fit(X, y)
print(clf.feature_log_prob_)
print(clf.class_log_prior_)

[[-0.69314718 -0.69314718]
 [-0.69314718 -0.69314718]]
[-0.69314718 -0.69314718]
